# Hyperparameter Tuning: Finding the Best Model Configuration

## What We'll Learn

In this notebook, we'll explore how to **systematically find the best hyperparameters** for machine learning models. While training adjusts model parameters (weights and biases), **hyperparameters** are the settings we choose before training begins — like learning rate, number of layers, regularization strength, etc.

**Key questions we'll answer:**
- What's the difference between parameters and hyperparameters?
- How do we search through possible configurations efficiently?
- Which hyperparameters have the biggest impact on performance?
- When should we use grid search vs. random search vs. Bayesian optimization?

**Why this matters:**
- The difference between a mediocre model and a great one often comes down to hyperparameter tuning
- Poor hyperparameters can make even the best architectures fail
- Efficient search strategies save time and computational resources

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_digits
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from itertools import product
import time
from scipy.stats import uniform, randint

# For reproducibility
np.random.seed(42)

## 2. Understanding Parameters vs. Hyperparameters

Before we dive into tuning, let's clarify the distinction:

**Parameters** are learned from data during training:
- Neural network weights
- Neural network biases
- Updated by gradient descent

**Hyperparameters** are set before training:
- Learning rate
- Number of layers/neurons
- Batch size
- Regularization strength
- Optimizer choice
- Number of training epochs

Think of hyperparameters as the "settings" that control how the learning process works.

## 3. Create a Simple Dataset

Let's use the digits dataset (8x8 images of handwritten digits 0-9) for our experiments. It's small enough to train quickly but complex enough that hyperparameters matter.

In [ ]:
# Load digits dataset
digits = load_digits()
X, y = digits.data, digits.target

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Feature range: [{X.min():.1f}, {X.max():.1f}]")

Split into train, validation, and test sets. We'll use the validation set for hyperparameter selection.

In [ ]:
# Split: 60% train, 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

Let's visualize a few examples to understand our data.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Baseline: Manual Search

Let's start by manually trying a few hyperparameter combinations. This helps us understand:
1. How sensitive the model is to hyperparameters
2. Why we need a systematic approach

In [ ]:
def train_and_evaluate(hidden_size, learning_rate, max_iter=100):
    """Train a simple MLP and return validation accuracy."""
    model = MLPClassifier(
        hidden_layer_sizes=(hidden_size,),
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        random_state=42,
        early_stopping=True
    )
    
    model.fit(X_train, y_train)
    val_acc = model.score(X_val, y_val)
    
    return val_acc

Try a few manual configurations:

In [ ]:
manual_configs = [
    {'hidden_size': 32, 'learning_rate': 0.01},
    {'hidden_size': 64, 'learning_rate': 0.001},
    {'hidden_size': 128, 'learning_rate': 0.1},
]

print("Manual hyperparameter search:\n")
for config in manual_configs:
    acc = train_and_evaluate(**config)
    print(f"Hidden: {config['hidden_size']:3d}, LR: {config['learning_rate']:.4f} → Val Acc: {acc:.4f}")

**Observation:** We see different configurations give different results, but this approach has problems:
- We might miss better combinations
- It's not systematic
- Hard to know if we've explored enough
- Doesn't scale to many hyperparameters

Let's explore systematic approaches!

## 5. Grid Search: Exhaustive Exploration

**Grid search** tests every combination of hyperparameters from predefined lists.

**How it works:**
1. Define a grid of values for each hyperparameter
2. Train a model for every combination
3. Pick the combination with best validation performance

**Pros:**
- Systematic and thorough
- Easy to understand and implement
- Guaranteed to find the best combination in the grid

**Cons:**
- Exponential growth in number of trials (curse of dimensionality)
- Wastes computation on unlikely regions
- Can't explore continuous spaces efficiently

### 5.1 Grid Search from Scratch

Let's implement grid search ourselves to understand how it works.

In [ ]:
# Define the grid
param_grid = {
    'hidden_size': [32, 64, 128],
    'learning_rate': [0.001, 0.01, 0.1],
}

# Generate all combinations
keys = param_grid.keys()
values = param_grid.values()
combinations = [dict(zip(keys, v)) for v in product(*values)]

print(f"Total combinations to try: {len(combinations)}")
print("\nFirst few combinations:")
for i, combo in enumerate(combinations[:3]):
    print(f"{i+1}. {combo}")

Now train models for all combinations and track results.

In [ ]:
grid_results = []

print("Running grid search...\n")
start_time = time.time()

for i, config in enumerate(combinations):
    acc = train_and_evaluate(**config)
    grid_results.append({
        'hidden_size': config['hidden_size'],
        'learning_rate': config['learning_rate'],
        'val_accuracy': acc
    })
    print(f"{i+1}/{len(combinations)}: Hidden={config['hidden_size']:3d}, LR={config['learning_rate']:.4f} → Acc={acc:.4f}")

elapsed = time.time() - start_time
print(f"\nGrid search completed in {elapsed:.2f} seconds")

Find the best configuration:

In [ ]:
best_result = max(grid_results, key=lambda x: x['val_accuracy'])

print("Best configuration found:")
print(f"  Hidden size: {best_result['hidden_size']}")
print(f"  Learning rate: {best_result['learning_rate']}")
print(f"  Validation accuracy: {best_result['val_accuracy']:.4f}")

### 5.2 Visualize the Search Space

Let's visualize how accuracy varies across the hyperparameter grid.

In [ ]:
# Reshape results into a grid for visualization
hidden_sizes = sorted(param_grid['hidden_size'])
learning_rates = sorted(param_grid['learning_rate'])

acc_grid = np.zeros((len(learning_rates), len(hidden_sizes)))

for result in grid_results:
    i = learning_rates.index(result['learning_rate'])
    j = hidden_sizes.index(result['hidden_size'])
    acc_grid[i, j] = result['val_accuracy']

# Plot heatmap
plt.figure(figsize=(10, 6))
im = plt.imshow(acc_grid, aspect='auto', cmap='RdYlGn', vmin=0.5, vmax=1.0)
plt.colorbar(im, label='Validation Accuracy')
plt.xticks(range(len(hidden_sizes)), hidden_sizes)
plt.yticks(range(len(learning_rates)), [f"{lr:.4f}" for lr in learning_rates])
plt.xlabel('Hidden Size')
plt.ylabel('Learning Rate')
plt.title('Grid Search Results: Validation Accuracy')

# Annotate cells with values
for i in range(len(learning_rates)):
    for j in range(len(hidden_sizes)):
        text = plt.text(j, i, f'{acc_grid[i, j]:.3f}',
                       ha="center", va="center", color="black", fontsize=9)

plt.tight_layout()
plt.show()

**Key insight:** Notice how some regions of the hyperparameter space perform much better than others. Learning rate has a particularly strong effect!

### 5.3 Using Scikit-learn's GridSearchCV

In practice, we can use built-in tools that also handle cross-validation automatically.

In [ ]:
param_grid_sklearn = {
    'hidden_layer_sizes': [(32,), (64,), (128,)],
    'learning_rate_init': [0.001, 0.01, 0.1],
    'alpha': [0.0001, 0.001, 0.01],  # L2 regularization
}

base_model = MLPClassifier(max_iter=100, random_state=42, early_stopping=True)

# GridSearchCV performs cross-validation for each configuration
grid_search = GridSearchCV(
    base_model,
    param_grid_sklearn,
    cv=3,  # 3-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,  # Use all CPU cores
    verbose=1
)

# Combine train and val for cross-validation
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.hstack([y_train, y_val])

print(f"\nSearching {len(param_grid_sklearn['hidden_layer_sizes']) * len(param_grid_sklearn['learning_rate_init']) * len(param_grid_sklearn['alpha'])} configurations with 3-fold CV...\n")
grid_search.fit(X_train_val, y_train_val)

Display the output.

In [ ]:
print("Best parameters found:")
print(grid_search.best_params_)
print(f"\nBest cross-validation score: {grid_search.best_score_:.4f}")
print(f"Test set score: {grid_search.score(X_test, y_test):.4f}")

## 6. Random Search: Smarter Exploration

**Random search** samples hyperparameters randomly from specified distributions.

**Key insight (Bergstra & Bengio, 2012):** When some hyperparameters matter more than others, random search explores the important dimensions more effectively than grid search with the same computational budget.

Consider this scenario:
- Grid: 3×3 grid = 9 trials, but only explores 3 unique values per dimension
- Random: 9 random samples = 9 unique values per dimension (likely)

### 6.1 Why Random Search Works

Let's visualize why random search can be more efficient.

In [ ]:
# Simulate a scenario where only one hyperparameter really matters
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grid search
grid_x = [1, 2, 3] * 3
grid_y = [1]*3 + [2]*3 + [3]*3
ax1.scatter(grid_x, grid_y, s=200, c='blue', alpha=0.6, edgecolors='black', linewidth=2)
ax1.set_xlim(0.5, 3.5)
ax1.set_ylim(0.5, 3.5)
ax1.set_xlabel('Important Hyperparameter', fontsize=12)
ax1.set_ylabel('Less Important Hyperparameter', fontsize=12)
ax1.set_title('Grid Search: 9 trials\nOnly 3 unique values for important param', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_xticks([1, 2, 3])
ax1.set_yticks([1, 2, 3])

# Random search
np.random.seed(42)
random_x = np.random.uniform(0.7, 3.3, 9)
random_y = np.random.uniform(0.7, 3.3, 9)
ax2.scatter(random_x, random_y, s=200, c='green', alpha=0.6, edgecolors='black', linewidth=2)
ax2.set_xlim(0.5, 3.5)
ax2.set_ylim(0.5, 3.5)
ax2.set_xlabel('Important Hyperparameter', fontsize=12)
ax2.set_ylabel('Less Important Hyperparameter', fontsize=12)
ax2.set_title('Random Search: 9 trials\n~9 unique values for important param', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key observation:** Random search explores more unique values along each dimension, increasing the chance of finding better configurations when we don't know which hyperparameters matter most.

### 6.2 Implement Random Search

Let's use scikit-learn's RandomizedSearchCV.

In [ ]:
# Define distributions to sample from
param_distributions = {
    'hidden_layer_sizes': [(h,) for h in [16, 32, 64, 128, 256]],
    'learning_rate_init': uniform(0.0001, 0.1),  # Sample uniformly from [0.0001, 0.1001]
    'alpha': uniform(0.00001, 0.01),  # L2 regularization
}

random_search = RandomizedSearchCV(
    base_model,
    param_distributions,
    n_iter=20,  # Number of random samples
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Running random search with 20 iterations...\n")
random_search.fit(X_train_val, y_train_val)

Display the output.

In [ ]:
print("Best parameters found by random search:")
print(random_search.best_params_)
print(f"\nBest cross-validation score: {random_search.best_score_:.4f}")
print(f"Test set score: {random_search.score(X_test, y_test):.4f}")

### 6.3 Compare Grid vs Random Search

Let's compare the exploration patterns and results.

In [ ]:
# Extract and compare results
print("Comparison Summary:\n")
print(f"Grid Search:")
print(f"  Configurations tried: {len(grid_search.cv_results_['params'])}")
print(f"  Best CV score: {grid_search.best_score_:.4f}")
print(f"  Test score: {grid_search.score(X_test, y_test):.4f}")

print(f"\nRandom Search:")
print(f"  Configurations tried: {len(random_search.cv_results_['params'])}")
print(f"  Best CV score: {random_search.best_score_:.4f}")
print(f"  Test score: {random_search.score(X_test, y_test):.4f}")

if random_search.best_score_ > grid_search.best_score_:
    print(f"\n✓ Random search found a better configuration!")
else:
    print(f"\n✓ Grid search found a better (or equal) configuration.")

Visualize the distribution of sampled learning rates:

In [ ]:
random_lrs = [params['learning_rate_init'] for params in random_search.cv_results_['params']]
random_scores = random_search.cv_results_['mean_test_score']

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(random_lrs, bins=15, alpha=0.7, color='green', edgecolor='black')
plt.xlabel('Learning Rate')
plt.ylabel('Count')
plt.title('Random Search: Learning Rate Distribution')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(random_lrs, random_scores, alpha=0.6, s=100, color='green', edgecolors='black')
plt.xlabel('Learning Rate')
plt.ylabel('CV Accuracy')
plt.title('Random Search: Learning Rate vs Performance')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Bayesian Optimization: Smart Sequential Search

**Bayesian optimization** is a sequential strategy that builds a probabilistic model of the objective function and uses it to decide where to search next.

**Key concepts:**
1. **Surrogate model** (often a Gaussian Process): Models the unknown objective function
2. **Acquisition function**: Decides where to sample next by balancing:
   - **Exploitation**: Sample where we expect high performance
   - **Exploration**: Sample where we're uncertain

**When to use it:**
- Expensive evaluations (training takes a long time)
- Continuous hyperparameter spaces
- You have a limited computational budget

**Trade-offs:**
- More sophisticated (harder to implement from scratch)
- Better sample efficiency
- Sequential (can't parallelize as easily)

### 7.1 Conceptual Example: How Bayesian Optimization Works

Let's visualize the key intuition with a simple 1D example.

In [ ]:
# Create a toy objective function (hidden from the optimizer)
def true_objective(x):
    return -(x - 0.7)**2 + 0.9 + 0.1 * np.sin(10 * x)

x_range = np.linspace(0, 1, 100)
y_true = true_objective(x_range)

# Simulate initial random samples
np.random.seed(42)
x_samples = np.array([0.1, 0.5, 0.9])
y_samples = true_objective(x_samples)

plt.figure(figsize=(12, 5))

# Plot true function
plt.plot(x_range, y_true, 'b-', linewidth=2, label='True objective (unknown)', alpha=0.3)
plt.scatter(x_samples, y_samples, s=200, c='red', zorder=5, label='Sampled points', edgecolors='black', linewidth=2)

# Highlight the best point so far
best_idx = np.argmax(y_samples)
plt.scatter(x_samples[best_idx], y_samples[best_idx], s=300, c='gold', marker='*', 
           zorder=6, label='Best so far', edgecolors='black', linewidth=2)

plt.xlabel('Hyperparameter Value', fontsize=12)
plt.ylabel('Performance', fontsize=12)
plt.title('Bayesian Optimization Intuition\nBalancing exploitation (near best) vs exploration (uncertain regions)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Bayesian optimization would:")
print("1. Build a model of the objective from these 3 samples")
print("2. Use uncertainty estimates to balance exploration vs exploitation")
print("3. Choose the next point to sample strategically")
print("4. Repeat until budget exhausted")

### 7.2 Bayesian Optimization with Optuna

Let's use the Optuna library for a practical example. Optuna uses Tree-structured Parzen Estimator (TPE) as its default Bayesian optimization algorithm.

In [ ]:
# Install optuna if needed
try:
    import optuna
except ImportError:
    print("Note: Optuna not installed. Install with: pip install optuna")
    print("Skipping Bayesian optimization example.")
    optuna = None

Execute the training loop.

In [ ]:
if optuna is not None:
    # Suppress optuna's verbose output
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    
    def objective(trial):
        # Suggest hyperparameters
        hidden_size = trial.suggest_int('hidden_size', 16, 256)
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
        alpha = trial.suggest_float('alpha', 1e-5, 1e-2, log=True)
        
        # Train and evaluate
        model = MLPClassifier(
            hidden_layer_sizes=(hidden_size,),
            learning_rate_init=learning_rate,
            alpha=alpha,
            max_iter=100,
            random_state=42,
            early_stopping=True
        )
        
        model.fit(X_train, y_train)
        val_acc = model.score(X_val, y_val)
        
        return val_acc
    
    # Create study and optimize
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    
    print("Running Bayesian optimization with Optuna (20 trials)...\n")
    study.optimize(objective, n_trials=20, show_progress_bar=True)
    
    print("\nBest parameters found by Bayesian optimization:")
    print(study.best_params)
    print(f"\nBest validation score: {study.best_value:.4f}")

Visualize the optimization progress:

In [ ]:
if optuna is not None:
    # Plot optimization history
    trials = study.trials
    values = [t.value for t in trials]
    best_so_far = np.maximum.accumulate(values)
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(values, 'o-', alpha=0.6, label='Trial value')
    plt.plot(best_so_far, 'r-', linewidth=2, label='Best so far')
    plt.xlabel('Trial')
    plt.ylabel('Validation Accuracy')
    plt.title('Bayesian Optimization Progress')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    lrs = [t.params['learning_rate'] for t in trials]
    plt.scatter(lrs, values, alpha=0.6, s=100, c=range(len(trials)), 
               cmap='viridis', edgecolors='black')
    plt.colorbar(label='Trial Number')
    plt.xlabel('Learning Rate (log scale)')
    plt.ylabel('Validation Accuracy')
    plt.title('Learning Rate Exploration')
    plt.xscale('log')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Notice how Bayesian optimization quickly focuses on promising regions!")

## 8. Which Hyperparameters Matter Most?

Not all hyperparameters are equally important. Let's analyze which ones have the biggest impact.

### 8.1 Learning Rate: The Most Critical Hyperparameter

Learning rate is almost always the most important hyperparameter. Let's demonstrate why.

In [ ]:
# Test a range of learning rates with fixed other hyperparameters
learning_rates_test = np.logspace(-5, 0, 15)  # 10^-5 to 10^0
lr_results = []

print("Testing learning rate sensitivity...\n")
for lr in learning_rates_test:
    model = MLPClassifier(
        hidden_layer_sizes=(64,),
        learning_rate_init=lr,
        alpha=0.001,
        max_iter=100,
        random_state=42,
        early_stopping=True
    )
    model.fit(X_train, y_train)
    acc = model.score(X_val, y_val)
    lr_results.append(acc)
    print(f"LR: {lr:.6f} → Val Acc: {acc:.4f}")

Visualize the results.

In [ ]:
plt.figure(figsize=(10, 6))
plt.semilogx(learning_rates_test, lr_results, 'o-', linewidth=2, markersize=8)
plt.xlabel('Learning Rate (log scale)', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.title('Learning Rate Sensitivity Analysis\nFixed: hidden_size=64, alpha=0.001', fontsize=12)
plt.grid(True, alpha=0.3)
plt.axhline(max(lr_results), color='red', linestyle='--', alpha=0.5, label=f'Best: {max(lr_results):.4f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nLearning rate range for good performance: {learning_rates_test[np.array(lr_results) > 0.95 * max(lr_results)].min():.6f} to {learning_rates_test[np.array(lr_results) > 0.95 * max(lr_results)].max():.6f}")

**Key observation:** Performance varies dramatically with learning rate:
- Too small → slow convergence, poor results
- Too large → instability, poor results
- Sweet spot → good performance

**Pro tip:** Always search learning rate on a **log scale** (e.g., 10^-5 to 10^-1) rather than linearly!

### 8.2 Architecture Hyperparameters

Network architecture (depth and width) is the next most important factor.

In [ ]:
# Test different hidden sizes with optimal learning rate
hidden_sizes_test = [8, 16, 32, 64, 128, 256]
optimal_lr = learning_rates_test[np.argmax(lr_results)]

arch_results = []
print(f"Testing architecture with optimal LR={optimal_lr:.6f}...\n")

for h in hidden_sizes_test:
    model = MLPClassifier(
        hidden_layer_sizes=(h,),
        learning_rate_init=optimal_lr,
        alpha=0.001,
        max_iter=100,
        random_state=42,
        early_stopping=True
    )
    model.fit(X_train, y_train)
    acc = model.score(X_val, y_val)
    arch_results.append(acc)
    print(f"Hidden size: {h:3d} → Val Acc: {acc:.4f}")

Visualize the results.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(hidden_sizes_test, arch_results, 'o-', linewidth=2, markersize=10, color='purple')
plt.xlabel('Hidden Layer Size', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.title(f'Architecture Sensitivity Analysis\nFixed: learning_rate={optimal_lr:.6f}, alpha=0.001', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBest hidden size: {hidden_sizes_test[np.argmax(arch_results)]} (accuracy: {max(arch_results):.4f})")
print(f"Performance is less sensitive to architecture than learning rate!")

### 8.3 Regularization Strength

Regularization (L2 penalty, controlled by `alpha`) prevents overfitting but is typically less critical.

In [ ]:
# Test different regularization strengths
alphas_test = np.logspace(-6, -1, 12)
reg_results = []

print("Testing regularization sensitivity...\n")
for alpha in alphas_test:
    model = MLPClassifier(
        hidden_layer_sizes=(64,),
        learning_rate_init=optimal_lr,
        alpha=alpha,
        max_iter=100,
        random_state=42,
        early_stopping=True
    )
    model.fit(X_train, y_train)
    acc = model.score(X_val, y_val)
    reg_results.append(acc)
    print(f"Alpha: {alpha:.7f} → Val Acc: {acc:.4f}")

Visualize the results.

In [ ]:
plt.figure(figsize=(10, 6))
plt.semilogx(alphas_test, reg_results, 'o-', linewidth=2, markersize=8, color='orange')
plt.xlabel('Regularization Strength (alpha, log scale)', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.title(f'Regularization Sensitivity Analysis\nFixed: hidden_size=64, learning_rate={optimal_lr:.6f}', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nPerformance variation: {max(reg_results) - min(reg_results):.4f}")
print(f"Compare to learning rate variation: {max(lr_results) - min(lr_results):.4f}")
print(f"\nRegularization matters less than learning rate for this problem.")

### 8.4 Hyperparameter Importance Ranking

Let's summarize what we've learned about hyperparameter importance.

In [ ]:
# Calculate sensitivity as the range of performance
sensitivities = {
    'Learning Rate': max(lr_results) - min(lr_results),
    'Hidden Size': max(arch_results) - min(arch_results),
    'Regularization': max(reg_results) - min(reg_results),
}

# Sort by sensitivity
sorted_params = sorted(sensitivities.items(), key=lambda x: x[1], reverse=True)

plt.figure(figsize=(10, 6))
params, sens = zip(*sorted_params)
colors = ['red', 'purple', 'orange']
bars = plt.bar(params, sens, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
plt.ylabel('Sensitivity\n(max - min performance)', fontsize=12)
plt.title('Hyperparameter Importance Ranking\nHigher = More Important to Tune', fontsize=12)
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, value in zip(bars, sens):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nGeneral Hyperparameter Importance (for most problems):\n")
print("1. ⭐ Learning rate - ALWAYS tune this first!")
print("2. ⭐ Architecture (depth, width) - Second priority")
print("3. Batch size - Affects training stability and speed")
print("4. Regularization strength - Important if overfitting")
print("5. Optimizer choice - Usually Adam is a good default")
print("6. Activation functions - Less critical, ReLU works well")

## 9. Practical Tips for Hyperparameter Tuning

### 9.1 Use Log Scale for Learning Rate and Regularization

These hyperparameters span multiple orders of magnitude, so search them on a log scale.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale (bad)
linear_lrs = np.linspace(0.0001, 0.1, 10)
ax1.plot(range(len(linear_lrs)), linear_lrs, 'o-', markersize=10)
ax1.set_xlabel('Sample Index')
ax1.set_ylabel('Learning Rate')
ax1.set_title('❌ Linear Scale\nMost samples concentrated in high values')
ax1.grid(True, alpha=0.3)
for i, lr in enumerate(linear_lrs[:5]):
    ax1.text(i, lr, f'{lr:.4f}', ha='center', va='bottom', fontsize=8)

# Log scale (good)
log_lrs = np.logspace(-4, -1, 10)
ax2.plot(range(len(log_lrs)), log_lrs, 'o-', markersize=10, color='green')
ax2.set_xlabel('Sample Index')
ax2.set_ylabel('Learning Rate')
ax2.set_title('✓ Log Scale\nSamples evenly distributed across orders of magnitude')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)
for i, lr in enumerate(log_lrs[:5]):
    ax2.text(i, lr, f'{lr:.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

### 9.2 Start with Wide Ranges, Then Narrow Down

Use a coarse search first to find promising regions, then refine.

In [ ]:
print("Two-stage search strategy:\n")
print("Stage 1: Coarse search")
print("  - Wide ranges (e.g., LR: 1e-5 to 1e-1)")
print("  - Fewer trials (e.g., 10-20)")
print("  - Identify promising regions\n")

print("Stage 2: Fine-grained search")
print("  - Narrow ranges around best from Stage 1")
print("  - More trials (e.g., 20-50)")
print("  - Find optimal configuration\n")

print("Example:")
print("  Stage 1 finds best LR ≈ 0.01")
print("  Stage 2 searches [0.005, 0.02] more densely")

### 9.3 Use Early Stopping During Search

Don't waste time training bad configurations to completion.

In [ ]:
print("Early stopping strategies during hyperparameter search:\n")
print("1. **Successive Halving**:")
print("   - Train many configs for few epochs")
print("   - Keep top 50%, train longer")
print("   - Repeat until one winner\n")

print("2. **Asynchronous Successive Halving (ASHA)**:")
print("   - Like successive halving but parallel")
print("   - Available in Ray Tune\n")

print("3. **Median Stopping**:")
print("   - Stop if trial is worse than median of others")
print("   - Available in Optuna\n")

print("Benefits:")
print("  ✓ 3-10x speedup in practice")
print("  ✓ More trials in same time budget")
print("  ✓ Better final hyperparameters")

### 9.4 Don't Forget About Computational Budget

Choose your search strategy based on available resources.

In [ ]:
recommendations = [
    ("Very limited (<10 trials)", "Manual search or Bayesian opt"),
    ("Limited (10-50 trials)", "Random search or Bayesian opt"),
    ("Moderate (50-200 trials)", "Random search"),
    ("Large (>200 trials)", "Grid search or random search"),
    ("Very expensive evaluations", "Bayesian optimization"),
]

print("Hyperparameter Search Strategy by Budget:\n")
for budget, strategy in recommendations:
    print(f"  {budget:30s} → {strategy}")

print("\n" + "="*60)
print("Rule of thumb: If each trial takes >1 hour, use Bayesian optimization")
print("="*60)

## 10. Final Test: Compare Best Models

Let's train final models with the best hyperparameters from each search method and compare on the test set.

In [ ]:
# Train with best parameters from each method
results_summary = {}

# Grid search best
grid_best = MLPClassifier(**grid_search.best_params_, max_iter=200, random_state=42)
grid_best.fit(X_train_val, y_train_val)
results_summary['Grid Search'] = grid_best.score(X_test, y_test)

# Random search best
random_best = MLPClassifier(**random_search.best_params_, max_iter=200, random_state=42)
random_best.fit(X_train_val, y_train_val)
results_summary['Random Search'] = random_best.score(X_test, y_test)

# Bayesian optimization best (if available)
if optuna is not None:
    bayes_best = MLPClassifier(
        hidden_layer_sizes=(study.best_params['hidden_size'],),
        learning_rate_init=study.best_params['learning_rate'],
        alpha=study.best_params['alpha'],
        max_iter=200,
        random_state=42
    )
    bayes_best.fit(X_train_val, y_train_val)
    results_summary['Bayesian Opt'] = bayes_best.score(X_test, y_test)

# Add baseline (default hyperparameters)
baseline = MLPClassifier(max_iter=200, random_state=42)
baseline.fit(X_train_val, y_train_val)
results_summary['Baseline (default)'] = baseline.score(X_test, y_test)

Create a bar chart to compare values.

In [ ]:
# Visualize final comparison
methods = list(results_summary.keys())
scores = list(results_summary.values())

plt.figure(figsize=(10, 6))
bars = plt.bar(methods, scores, color=['blue', 'green', 'purple', 'gray'][:len(methods)], 
              alpha=0.7, edgecolor='black', linewidth=2)
plt.ylabel('Test Set Accuracy', fontsize=12)
plt.title('Final Model Comparison: Test Set Performance', fontsize=12)
plt.ylim([min(scores) - 0.05, 1.0])
plt.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, score in zip(bars, scores):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nFinal Test Set Results:\n")
for method, score in results_summary.items():
    improvement = (score - results_summary['Baseline (default)']) / results_summary['Baseline (default)'] * 100
    print(f"{method:20s}: {score:.4f} ({improvement:+.1f}% vs baseline)")

## 11. Key Takeaways

### Core Concepts

1. **Parameters vs Hyperparameters**
   - Parameters are learned during training (weights, biases)
   - Hyperparameters are set before training (learning rate, architecture, etc.)

2. **Search Strategies**
   - **Manual search**: Good for understanding, doesn't scale
   - **Grid search**: Exhaustive but exponentially expensive
   - **Random search**: Often better than grid with same budget
   - **Bayesian optimization**: Best for expensive evaluations

3. **Hyperparameter Importance**
   - Learning rate is almost always most critical (search on log scale!)
   - Architecture (depth/width) comes second
   - Other hyperparameters are often less sensitive

### Practical Guidelines

1. **Always tune learning rate first** using a log scale (e.g., 1e-5 to 1e-1)
2. **Use random search** as your default strategy (better than grid for same budget)
3. **Use Bayesian optimization** when evaluations are expensive (>1 hour per trial)
4. **Search in two stages**: coarse (wide range) → fine (narrow range)
5. **Use early stopping** to avoid wasting time on bad configurations
6. **Always evaluate on a held-out test set** after hyperparameter selection

### When to Use Each Strategy

| Budget | Best Strategy |
|--------|---------------|
| <10 trials | Manual or Bayesian |
| 10-50 trials | Random or Bayesian |
| 50-200 trials | Random |
| >200 trials | Grid or Random |
| Expensive evaluations | Bayesian |

### Remember

> "A mediocre algorithm with great hyperparameters often beats a great algorithm with mediocre hyperparameters."

> "Always search learning rate on a log scale — it's the most important hyperparameter and spans many orders of magnitude."

> "Random search often beats grid search because it explores more unique values along important dimensions."

## 12. Next Steps

Now that you understand hyperparameter tuning, you can:

1. **Apply to your own models**: Use these techniques on any ML model
2. **Learn advanced techniques**: Look into ASHA (Asynchronous Successive Halving), Hyperband
3. **Explore AutoML**: Check out tools like Ray Tune, Optuna, Weights & Biases Sweeps
4. **Study learning rate schedules**: Combine good initial LR with adaptive schedules
5. **Understand neural architecture search (NAS)**: Automated architecture design

**Tools to explore:**
- **Ray Tune**: Scalable hyperparameter tuning with advanced schedulers
- **Optuna**: Easy-to-use Bayesian optimization
- **Weights & Biases Sweeps**: Hyperparameter tuning with great visualization
- **scikit-optimize**: Bayesian optimization for scikit-learn

Happy tuning! 🎯